<a href="https://colab.research.google.com/github/AabidMK/CricketIQ_Infosys_Internship_Feb2025/blob/Mudra-Bhavya-Sri/Task4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files

uploaded = files.upload()  # This will prompt you to upload files manually


Saving matches (2).csv to matches (2).csv


In [ ]:
from google.colab import files

uploaded = files.upload()  # This will prompt you to upload files manually


Saving deliveries.csv (1).zip to deliveries.csv (1).zip


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
import zipfile

In [ ]:
# Load matches dataset
matches_df = pd.read_csv('matches (2).csv')


In [ ]:
# Unzip deliveries dataset and load it
with zipfile.ZipFile('deliveries.csv (1).zip', 'r') as zip_ref:
    zip_ref.extractall()  # Extracts the files in the same directory


In [ ]:
# Now read the extracted deliveries.csv
deliveries_df = pd.read_csv('deliveries.csv')


In [ ]:
# Display first few rows to confirm loading
print("Matches Data:")
print(matches_df.head())
print("\nDeliveries Data:")
print(deliveries_df.head())

Matches Data:
       id   season        city        date match_type player_of_match  \
0  335982  2007/08   Bangalore  18-04-2008     League     BB McCullum   
1  335983  2007/08  Chandigarh  19-04-2008     League      MEK Hussey   
2  335984  2007/08       Delhi  19-04-2008     League     MF Maharoof   
3  335985  2007/08      Mumbai  20-04-2008     League      MV Boucher   
4  335986  2007/08     Kolkata  20-04-2008     League       DJ Hussey   

                                        venue                        team1  \
0                       M Chinnaswamy Stadium  Royal Challengers Bangalore   
1  Punjab Cricket Association Stadium, Mohali              Kings XI Punjab   
2                            Feroz Shah Kotla             Delhi Daredevils   
3                            Wankhede Stadium               Mumbai Indians   
4                                Eden Gardens        Kolkata Knight Riders   

                         team2                  toss_winner toss_decision  \
0

In [ ]:
# Extract relevant columns from matches dataset
matches_df_selected = matches_df[[
    'id', 'date', 'team1', 'team2', 'toss_winner', 'toss_decision',
    'winner', 'result', 'result_margin'
]]

# Extract relevant columns from deliveries dataset
deliveries_selected = deliveries_df[[
    'match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball',
    'total_runs', 'is_wicket'
]]

# Merge datasets on match ID
merged_df = deliveries_selected.merge(matches_df_selected, left_on='match_id', right_on='id', how='left')
merged_df.drop(columns=['id'], inplace=True)

# Create new features
merged_df['cumulative_runs'] = merged_df.groupby(['match_id', 'inning'])['total_runs'].cumsum()
merged_df['cumulative_wickets'] = merged_df.groupby(['match_id', 'inning'])['is_wicket'].cumsum()
merged_df['overs_completed'] = merged_df['over'] + (merged_df['ball'] / 6)
merged_df['current_run_rate'] = merged_df['cumulative_runs'] / merged_df['overs_completed']


In [ ]:
# ---- Calculate Required Run Rate (RRR) ----
# Step 1: Get the target runs (1st innings total + 1)
target_runs = merged_df[merged_df['inning'] == 1].groupby('match_id')['cumulative_runs'].max() + 1
target_runs = target_runs.rename('target_runs')

# Step 2: Merge target runs into 2nd innings data
merged_df = merged_df.merge(target_runs, on='match_id', how='left')

# Step 3: Calculate remaining overs (assuming T20 format with 20 overs)
merged_df['remaining_overs'] = 20 - merged_df['overs_completed']

# Step 4: Calculate RRR
merged_df['required_run_rate'] = (merged_df['target_runs'] - merged_df['cumulative_runs']) / merged_df['remaining_overs']


In [ ]:
# ---- Display the first rows to confirm changes ----
print("\nFirst 5 rows of the dataset with Required Run Rate:")
print(merged_df.head())


First 5 rows of the dataset with Required Run Rate:
   match_id  inning           batting_team                 bowling_team  over  \
0    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
1    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
2    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
3    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
4    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   

   ball  total_runs  is_wicket        date                        team1  ...  \
0     1           1          0  18-04-2008  Royal Challengers Bangalore  ...   
1     2           0          0  18-04-2008  Royal Challengers Bangalore  ...   
2     3           1          0  18-04-2008  Royal Challengers Bangalore  ...   
3     4           0          0  18-04-2008  Royal Challengers Bangalore  ...   
4     5           0          0  18-04-2008  Royal Challenger

In [ ]:
# Check for null values in the merged dataset
print("Missing values before preprocessing:")
print(merged_df.isnull().sum())

# Handle missing values
merged_df.fillna(0, inplace=True)  # Filling NaN values with 0 (modify based on your needs)

# Check for duplicate records
duplicates_count = merged_df.duplicated().sum()
print("\nDuplicate records before preprocessing:", duplicates_count)

# Remove duplicates if found
if duplicates_count > 0:
    merged_df.drop_duplicates(inplace=True)
    print("\nDuplicates removed!")

# Verify after preprocessing
print("\nMissing values after preprocessing:")
print(merged_df.isnull().sum())

print("\nDuplicate records after preprocessing:", merged_df.duplicated().sum())

# Display the first few rows after cleaning
print("\nCleaned dataset:")
print(merged_df.head())


Missing values before preprocessing:
match_id                 0
inning                   0
batting_team             0
bowling_team             0
over                     0
ball                     0
total_runs               0
is_wicket                0
date                     0
team1                    0
team2                    0
toss_winner              0
toss_decision            0
winner                 490
result                   0
result_margin         4124
cumulative_runs          0
cumulative_wickets       0
overs_completed          0
current_run_rate         0
target_runs              0
remaining_overs          0
required_run_rate       16
dtype: int64

Duplicate records before preprocessing: 0

Missing values after preprocessing:
match_id              0
inning                0
batting_team          0
bowling_team          0
over                  0
ball                  0
total_runs            0
is_wicket             0
date                  0
team1                 0
team2    

In [ ]:
print(merged_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 260920 entries, 0 to 260919
Data columns (total 23 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   match_id            260920 non-null  int64  
 1   inning              260920 non-null  int64  
 2   batting_team        260920 non-null  object 
 3   bowling_team        260920 non-null  object 
 4   over                260920 non-null  int64  
 5   ball                260920 non-null  int64  
 6   total_runs          260920 non-null  int64  
 7   is_wicket           260920 non-null  int64  
 8   date                260920 non-null  object 
 9   team1               260920 non-null  object 
 10  team2               260920 non-null  object 
 11  toss_winner         260920 non-null  object 
 12  toss_decision       260920 non-null  object 
 13  winner              260920 non-null  object 
 14  result              260920 non-null  object 
 15  result_margin       260920 non-nul

In [ ]:
# Rename team1 and team2 before replacing names
merged_df.rename(columns={'team1': 'home_team', 'team2': 'away_team'}, inplace=True)

# Dictionary of old team names to new team names
team_name_changes = {
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
    'Delhi Daredevils': 'Delhi Capitals',
    'Rising Pune Supergiants': 'Rising Pune Supergiants'
}

# Replace team names in all relevant columns
merged_df.replace({'batting_team': team_name_changes,
                   'bowling_team': team_name_changes,
                   'home_team': team_name_changes,
                   'away_team': team_name_changes}, inplace=True)

# ---- Display the first rows to confirm changes ----
print("\nUpdated Team Names in Dataset:")
print(merged_df[['batting_team', 'bowling_team', 'home_team', 'away_team']].drop_duplicates())



Updated Team Names in Dataset:
                       batting_team                 bowling_team  \
0             Kolkata Knight Riders  Royal Challengers Bengaluru   
124     Royal Challengers Bengaluru        Kolkata Knight Riders   
225             Chennai Super Kings              Kings XI Punjab   
349                 Kings XI Punjab          Chennai Super Kings   
473                Rajasthan Royals               Delhi Capitals   
...                             ...                          ...   
243723          Chennai Super Kings               Gujarat Titans   
247261         Lucknow Super Giants  Royal Challengers Bengaluru   
247387  Royal Challengers Bengaluru         Lucknow Super Giants   
256715        Kolkata Knight Riders         Lucknow Super Giants   
256844         Lucknow Super Giants        Kolkata Knight Riders   

                          home_team                    away_team  
0       Royal Challengers Bengaluru        Kolkata Knight Riders  
124     Royal Cha

In [ ]:
# Check for null values
print("\n Checking for Null Values in the Dataset:")
print(merged_df.isnull().sum())



 Checking for Null Values in the Dataset:
match_id              0
inning                0
batting_team          0
bowling_team          0
over                  0
ball                  0
total_runs            0
is_wicket             0
date                  0
home_team             0
away_team             0
toss_winner           0
toss_decision         0
winner                0
result                0
result_margin         0
cumulative_runs       0
cumulative_wickets    0
overs_completed       0
current_run_rate      0
target_runs           0
remaining_overs       0
required_run_rate     0
dtype: int64


In [ ]:
print(merged_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 260920 entries, 0 to 260919
Data columns (total 23 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   match_id            260920 non-null  int64  
 1   inning              260920 non-null  int64  
 2   batting_team        260920 non-null  object 
 3   bowling_team        260920 non-null  object 
 4   over                260920 non-null  int64  
 5   ball                260920 non-null  int64  
 6   total_runs          260920 non-null  int64  
 7   is_wicket           260920 non-null  int64  
 8   date                260920 non-null  object 
 9   home_team           260920 non-null  object 
 10  away_team           260920 non-null  object 
 11  toss_winner         260920 non-null  object 
 12  toss_decision       260920 non-null  object 
 13  winner              260920 non-null  object 
 14  result              260920 non-null  object 
 15  result_margin       260920 non-nul